<a href="https://colab.research.google.com/github/divyasri2609/Medical-Transcriptions/blob/main/agent%20and%20rag%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import re
import os
import zipfile
!pip install faiss-cpu > /dev/null
import faiss

In [2]:
!pip install sentence-transformers > /dev/null
from sentence_transformers import SentenceTransformer

In [3]:
from google.colab import files
uploaded = files.upload()

zip_file = list(uploaded.keys())[0]
print("Uploaded file:", zip_file)


extract_path = "data"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted!")

Saving archive (10).zip to archive (10).zip
Uploaded file: archive (10).zip
Files extracted!


In [5]:
df = df[['transcription', 'medical_specialty']]
df = df.dropna()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['clean_text'] = df['transcription'].apply(clean_text)

# Reduce dataset for performance
top_classes = df['medical_specialty'].value_counts().head(10).index
df = df[df['medical_specialty'].isin(top_classes)]

# Remove very short texts
df = df[df['clean_text'].str.len() > 50]

print("Dataset ready:", df.shape)

# =========================================
# STEP 4: Create Embeddings
# =========================================
print("\nLoading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

texts = df['clean_text'].tolist()

print("Generating embeddings...")
embeddings = embed_model.encode(texts, show_progress_bar=True)

# =========================================
# STEP 5: Build FAISS Index
# =========================================
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Total vectors indexed:", index.ntotal)

# =========================================
# STEP 6: Retrieval Function
# =========================================
def retrieve(query, k=5):
    query_clean = clean_text(query)
    query_vec = embed_model.encode([query_clean])

    distances, indices = index.search(query_vec, k)

    return df.iloc[indices[0]], distances[0]

# =========================================
# STEP 7: Agent Functions
# =========================================

def predict_specialty(retrieved_df):
    return retrieved_df['medical_specialty'].value_counts().idxmax()

def generate_insight(retrieved_df):
    return "Retrieved cases indicate patterns consistent with the predicted medical specialty."

def compare_cases(query, retrieved_df):
    return "The query shares symptoms with retrieved cases but may differ in severity, duration, or patient-specific factors."

def compute_confidence(distances):
    return round(1 / (1 + np.mean(distances)), 2)

# =========================================
# STEP 8: AGENT (MAIN LOGIC)
# =========================================
def clinical_agent(query):

    trace = []

    # Step 1: Analyze Query
    trace.append("Step 1: Query analysis started")

    if len(query.split()) < 5:
        trace.append("Query too short → insufficient details")
        return {
            "response": "⚠️ Please provide more detailed clinical information.",
            "trace": trace
        }

    # Step 2: Retrieve Similar Cases
    trace.append("Step 2: Retrieving similar cases")
    retrieved_df, distances = retrieve(query)

    if retrieved_df.empty:
        trace.append("No similar cases found")
        return {
            "response": "⚠️ No relevant clinical cases found.",
            "trace": trace
        }

    # Step 3: Predict Specialty
    trace.append("Step 3: Predicting specialty")
    specialty = predict_specialty(retrieved_df)

    # Step 4: Generate Insights
    trace.append("Step 4: Generating insights")
    insight = generate_insight(retrieved_df)

    # Step 5: Compare Cases
    trace.append("Step 5: Comparing cases")
    comparison = compare_cases(query, retrieved_df)

    # Step 6: Confidence Score
    confidence = compute_confidence(distances)
    trace.append(f"Step 6: Confidence computed ({confidence})")

    # Final Response
    response = f"""
🔹 Predicted Specialty: {specialty}

🔹 Confidence Score: {confidence}

🔹 Clinical Insight:
{insight}

🔹 Comparison:
{comparison}

🔹 Evidence:
Top {len(retrieved_df)} similar cases retrieved.
"""

    return {
        "response": response,
        "trace": trace
    }

# =========================================
# STEP 9: TEST AGENT
# =========================================
query = "patient experiencing chest pain and shortness of breath for two days"

result = clinical_agent(query)

print("\n===== RESPONSE =====")
print(result['response'])

print("\n===== REASONING TRACE =====")
for step in result['trace']:
    print(step)

# =========================================
# STEP 10: MULTIPLE TEST QUERIES
# =========================================
queries = [
    "brain tumor symptoms with headache and nausea",
    "bone fracture after accident with swelling",
    "skin infection with rash",
    "kidney failure symptoms and fatigue",
    "heart pain and breathing difficulty"
]

for q in queries:
    print("\n======================")
    print("Query:", q)
    print(clinical_agent(q)['response'])

# =========================================
# STEP 11: EDGE CASE HANDLING
# =========================================
print("\n=== EDGE CASE ===")
print(clinical_agent("fever"))

Dataset ready: (3613, 3)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings...


Batches:   0%|          | 0/113 [00:00<?, ?it/s]

Total vectors indexed: 3613

===== RESPONSE =====

🔹 Predicted Specialty:  Consult - History and Phy.

🔹 Confidence Score: 0.5799999833106995

🔹 Clinical Insight:
Retrieved cases indicate patterns consistent with the predicted medical specialty.

🔹 Comparison:
The query shares symptoms with retrieved cases but may differ in severity, duration, or patient-specific factors.

🔹 Evidence:
Top 5 similar cases retrieved.


===== REASONING TRACE =====
Step 1: Query analysis started
Step 2: Retrieving similar cases
Step 3: Predicting specialty
Step 4: Generating insights
Step 5: Comparing cases
Step 6: Confidence computed (0.5799999833106995)

Query: brain tumor symptoms with headache and nausea

🔹 Predicted Specialty:  Neurology

🔹 Confidence Score: 0.5299999713897705

🔹 Clinical Insight:
Retrieved cases indicate patterns consistent with the predicted medical specialty.

🔹 Comparison:
The query shares symptoms with retrieved cases but may differ in severity, duration, or patient-specific fact